# Prepare Augmented Data: MALLS v0.1 + Target Dataset

## Muc tieu
1. Load & inspect MALLS v0.1 external data (train 27K + test 1K = 28K sentence-level NL-FOL pairs)
2. Load & inspect Target processed data (multi-premise records)
3. Chuan hoa MALLS ve cung format voi Target
4. Gop train+test MALLS thanh 1 file duy nhat de pretrain (Stage 1)

## Chien luoc train (2-stage curriculum learning)
- **Stage 1**: Pretrain tren MALLS v0.1 gop (hoc NL->FOL tong quat, ko can do hieu suat)
- **Stage 2**: Fine-tune tren Target (hoc format JSON, multi-premise, mixed naming)

Ly do: MALLS 28K >> Target 630 → gop 1 lan se khien target bi 'chim'.

In [20]:
import json
import pandas as pd
import ast
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path("..").resolve()
MALLS_TRAIN_PATH = PROJECT_ROOT / "data" / "external" / "MALLS-v0" / "MALLS-v0.1-train.json"
MALLS_TEST_PATH = PROJECT_ROOT / "data" / "external" / "MALLS-v0" / "MALLS-v0.1-test.json"
TARGET_TRAIN = PROJECT_ROOT / "data" / "processed" / "train.csv"
TARGET_DEV = PROJECT_ROOT / "data" / "processed" / "dev.csv"
TARGET_TEST = PROJECT_ROOT / "data" / "processed" / "test.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("MALLS train:", MALLS_TRAIN_PATH, "| exists:", MALLS_TRAIN_PATH.exists())
print("MALLS test: ", MALLS_TEST_PATH, "| exists:", MALLS_TEST_PATH.exists())

Project root: E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project
MALLS train: E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\data\external\MALLS-v0\MALLS-v0.1-train.json | exists: True
MALLS test:  E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\data\external\MALLS-v0\MALLS-v0.1-test.json | exists: True


## 1. Load & Inspect MALLS v0.1 (External)

In [21]:
with open(MALLS_TRAIN_PATH, "r", encoding="utf-8") as f:
    malls_train_raw = json.load(f)
with open(MALLS_TEST_PATH, "r", encoding="utf-8") as f:
    malls_test_raw = json.load(f)

print(f"MALLS v0.1 train: {len(malls_train_raw)} samples")
print(f"MALLS v0.1 test:  {len(malls_test_raw)} samples")
print(f"Keys: {malls_train_raw[0].keys()}")
print()

# Stats (gom ca train + test de thong ke)
malls_all_raw = malls_train_raw + malls_test_raw
has_forall = sum(1 for x in malls_all_raw if "∀" in x["FOL"])
has_exists = sum(1 for x in malls_all_raw if "∃" in x["FOL"])
has_xor = sum(1 for x in malls_all_raw if "⊕" in x["FOL"])
has_iff = sum(1 for x in malls_all_raw if "↔" in x["FOL"])
has_const = sum(1 for x in malls_all_raw if "∀" not in x["FOL"] and "∃" not in x["FOL"])

print(f"∀ (forall): {has_forall}")
print(f"∃ (exists): {has_exists}")
print(f"⊕ (xor):    {has_xor}")
print(f"↔ (iff):    {has_iff}")
print(f"Ground atoms (no quantifier): {has_const}")
print()

print("=" * 60)
print("MALLS v0.1 TRAIN SAMPLES (truoc chuan hoa):")
print("=" * 60)
for i in range(5):
    print(f"\n--- Train sample {i} ---")
    print(f"  NL:  {malls_train_raw[i]['NL']}")
    print(f"  FOL: {malls_train_raw[i]['FOL']}")

print()
print("=" * 60)
print("MALLS v0.1 TEST SAMPLES (truoc chuan hoa):")
print("=" * 60)
for i in range(3):
    print(f"\n--- Test sample {i} ---")
    print(f"  NL:  {malls_test_raw[i]['NL']}")
    print(f"  FOL: {malls_test_raw[i]['FOL']}")

MALLS v0.1 train: 27284 samples
MALLS v0.1 test:  1000 samples
Keys: dict_keys(['NL', 'FOL'])

∀ (forall): 27256
∃ (exists): 1738
⊕ (xor):    1777
↔ (iff):    1835
Ground atoms (no quantifier): 193

MALLS v0.1 TRAIN SAMPLES (truoc chuan hoa):

--- Train sample 0 ---
  NL:  A film can be a drama, have a long runtime, and win multiple awards, or it can be a comedy, have a shorter runtime, and be a box office success.
  FOL: ∃x (Film(x) ∧ ((Drama(x) ∧ LongRuntime(x) ∧ MultipleAwards(x)) ∨ (Comedy(x) ∧ ShorterRuntime(x) ∧ BoxOfficeSuccess(x))))

--- Train sample 1 ---
  NL:  If a person is a librarian, they either work in a public library or an academic library.
  FOL: ∀x (Person(x) ∧ Librarian(x) → WorkInPublicLibrary(x) ⊕ WorkInAcademicLibrary(x))

--- Train sample 2 ---
  NL:  Healthy sleep habits improve overall well-being.
  FOL: ∀x (HealthySleepHabits(x) → ImprovesWellBeing(x))

--- Train sample 3 ---
  NL:  A shape can have three or four sides, but not both.
  FOL: ∀x (Shape(x) → (T

## 2. Load & Inspect Target (Processed)

In [22]:
df_train = pd.read_csv(TARGET_TRAIN)
df_dev = pd.read_csv(TARGET_DEV)
df_test = pd.read_csv(TARGET_TEST)

print(f"Target train: {len(df_train)} rows")
print(f"Target dev:   {len(df_dev)} rows")
print(f"Target test:  {len(df_test)} rows")
print(f"Columns: {list(df_train.columns)}")
print()

# Count unique records (1 record = 1 premise set, nhieu questions)
n_records_train = df_train["record_id"].nunique()
n_records_dev = df_dev["record_id"].nunique()
n_records_test = df_test["record_id"].nunique()
print(f"Unique records: train={n_records_train}, dev={n_records_dev}, test={n_records_test}")
print()

print("=" * 60)
print("TARGET SAMPLES (truoc chuan hoa):")
print("=" * 60)
for i in range(3):
    row = df_train.iloc[i]
    nl = ast.literal_eval(row["premises_nl"])
    fol = ast.literal_eval(row["premises_fol"]) if pd.notna(row["premises_fol"]) else []
    print(f"\n--- Record {row['record_id']}, Q{row['q_idx']} ---")
    print(f"  #premises: {len(nl)}")
    for j, (n, f_) in enumerate(zip(nl, fol)):
        print(f"  [{j}] NL:  {n}")
        print(f"      FOL: {f_}")

Target train: 645 rows
Target dev:   81 rows
Target test:  82 rows
Columns: ['record_id', 'q_idx', 'n_premises_used', 'question', 'answer', 'explanation', 'premises_nl', 'premises_fol']

Unique records: train=328, dev=41, test=42

TARGET SAMPLES (truoc chuan hoa):

--- Record 1, Q0 ---
  #premises: 11
  [0] NL:  Students who have completed the core curriculum and passed the science assessment are qualified for advanced courses.
      FOL: ∀x ((completed_core_curriculum(x) ∧ passed_science_assessment(x)) → qualified_for_advanced_courses(x))
  [1] NL:  Students who are qualified for advanced courses and have completed research methodology are eligible for the international program.
      FOL: ∀x ((qualified_for_advanced_courses(x) ∧ completed_research_methodology(x)) → eligible_for_international_program(x))
  [2] NL:  Students who have passed the language proficiency exam are eligible for the international program.
      FOL: ∀x (passed_language_proficiency(x) → eligible_for_internationa

## 3. Chuan hoa MALLS v0.1 ve format Target

MALLS format: `{NL: str, FOL: str}` (1 cap)

Target format: `premises_nl: list[str], premises_fol: list[str]` (nhieu cap)

Chuyen MALLS: moi sample thanh `{premises_nl: [NL], premises_fol: [FOL]}` (list 1 phan tu)

In [23]:
import re


def normalize_malls_constants(fol: str) -> str:
    """Chuan hoa constants trong MALLS: lowercase -> Capitalized.
    
    MALLS: helen, eiffelTower, paris
    Target: John, Alex, curriculum
    
    Rule: arguments trong ngoac don () ma la constant (khong phai variable x,y,z)
    -> Capitalize chu cai dau.
    """
    def _capitalize_const(match):
        name = match.group(1)
        # Variable 1 ky tu (x, y, z, s, c, f, e) -> giu nguyen
        if len(name) == 1:
            return f"({name}"
        # camelCase constant -> Capitalize first letter
        return f"({name[0].upper() + name[1:]}"
    
    # Match opening paren + lowercase word that looks like a constant
    result = re.sub(r"\(([a-z][a-zA-Z_]+)", _capitalize_const, fol)
    
    # Also handle constants after comma: ", helen)" -> ", Helen)"
    def _capitalize_after_comma(match):
        name = match.group(1)
        if len(name) == 1:
            return f", {name}"
        return f", {name[0].upper() + name[1:]}"
    
    result = re.sub(r",\s*([a-z][a-zA-Z_]+)", _capitalize_after_comma, result)
    return result


def malls_to_target_format(malls_samples: list[dict]) -> list[dict]:
    """Convert MALLS sang format tuong thich voi Target dataset."""
    converted = []
    skipped = 0
    for sample in malls_samples:
        nl = sample["NL"].strip()
        fol = sample["FOL"].strip()
        
        # Skip empty
        if not nl or not fol:
            skipped += 1
            continue
        
        # Normalize constants
        fol_normalized = normalize_malls_constants(fol)
        
        converted.append({
            "premises_nl": [nl],
            "premises_fol": [fol_normalized],
            "source": "MALLS",
        })
    
    print(f"Converted: {len(converted)} | Skipped: {skipped}")
    return converted


# --- Normalize & gop MALLS v0.1 train + test ---
print("=== MALLS v0.1 TRAIN ===")
malls_train_converted = malls_to_target_format(malls_train_raw)

print("\n=== MALLS v0.1 TEST ===")
malls_test_converted = malls_to_target_format(malls_test_raw)

# Gop thanh 1 list duy nhat
malls_converted = malls_train_converted + malls_test_converted
print(f"\n=== GOM LAI: {len(malls_converted)} samples (train {len(malls_train_converted)} + test {len(malls_test_converted)}) ===")

print()
print("=" * 60)
print("MALLS v0.1 SAMPLES (sau chuan hoa, gop):")
print("=" * 60)
for i in range(5):
    s = malls_converted[i]
    print(f"\n--- Sample {i} ---")
    print(f"  NL:  {s['premises_nl']}")
    print(f"  FOL: {s['premises_fol']}")

=== MALLS v0.1 TRAIN ===
Converted: 27284 | Skipped: 0

=== MALLS v0.1 TEST ===
Converted: 1000 | Skipped: 0

=== GOM LAI: 28284 samples (train 27284 + test 1000) ===

MALLS v0.1 SAMPLES (sau chuan hoa, gop):

--- Sample 0 ---
  NL:  ['A film can be a drama, have a long runtime, and win multiple awards, or it can be a comedy, have a shorter runtime, and be a box office success.']
  FOL: ['∃x (Film(x) ∧ ((Drama(x) ∧ LongRuntime(x) ∧ MultipleAwards(x)) ∨ (Comedy(x) ∧ ShorterRuntime(x) ∧ BoxOfficeSuccess(x))))']

--- Sample 1 ---
  NL:  ['If a person is a librarian, they either work in a public library or an academic library.']
  FOL: ['∀x (Person(x) ∧ Librarian(x) → WorkInPublicLibrary(x) ⊕ WorkInAcademicLibrary(x))']

--- Sample 2 ---
  NL:  ['Healthy sleep habits improve overall well-being.']
  FOL: ['∀x (HealthySleepHabits(x) → ImprovesWellBeing(x))']

--- Sample 3 ---
  NL:  ['A shape can have three or four sides, but not both.']
  FOL: ['∀x (Shape(x) → (ThreeSides(x) ⊕ FourSides(x))

In [24]:
# So sanh truoc/sau normalize constants
print("Constant normalization examples:")
print()
test_cases = [
    'Vegetarian(helen) ∧ ¬Vegan(helen)',
    'FamousLandmark(eiffelTower) ∧ In(eiffelTower, paris)',
    '∀x (Dog(x) ∧ WellTrained(x) → TherapyAnimal(x))',
    'FastestLandAnimal(cheetah)',
]
for tc in test_cases:
    print(f"  Before: {tc}")
    print(f"  After:  {normalize_malls_constants(tc)}")
    print()

Constant normalization examples:

  Before: Vegetarian(helen) ∧ ¬Vegan(helen)
  After:  Vegetarian(Helen) ∧ ¬Vegan(Helen)

  Before: FamousLandmark(eiffelTower) ∧ In(eiffelTower, paris)
  After:  FamousLandmark(EiffelTower) ∧ In(EiffelTower, Paris)

  Before: ∀x (Dog(x) ∧ WellTrained(x) → TherapyAnimal(x))
  After:  ∀x (Dog(x) ∧ WellTrained(x) → TherapyAnimal(x))

  Before: FastestLandAnimal(cheetah)
  After:  FastestLandAnimal(Cheetah)



## 4. Tao Target format tuong tu

Chuyen Target CSV thanh list[dict] de so sanh cung format.

In [25]:
def csv_to_target_format(df: pd.DataFrame) -> list[dict]:
    """Chuyen CSV thanh list[dict] voi premises_nl, premises_fol.
    
    Moi record_id chi xuat hien 1 lan (deduplicate theo record_id).
    """
    seen = set()
    records = []
    for _, row in df.iterrows():
        rid = row["record_id"]
        if rid in seen:
            continue
        seen.add(rid)
        nl = ast.literal_eval(row["premises_nl"])
        fol = ast.literal_eval(row["premises_fol"]) if pd.notna(row["premises_fol"]) else []
        if nl and fol and len(nl) == len(fol):
            records.append({
                "premises_nl": nl,
                "premises_fol": fol,
                "source": "target",
            })
    return records


target_train = csv_to_target_format(df_train)
target_dev = csv_to_target_format(df_dev)
target_test = csv_to_target_format(df_test)

print(f"Target unique records: train={len(target_train)}, dev={len(target_dev)}, test={len(target_test)}")
print()

print("=" * 60)
print("TARGET SAMPLES (format chuan):")
print("=" * 60)
for i in range(2):
    s = target_train[i]
    print(f"\n--- Record {i} ({len(s['premises_nl'])} premises) ---")
    for j in range(min(3, len(s['premises_nl']))):
        print(f"  [{j}] NL:  {s['premises_nl'][j]}")
        print(f"      FOL: {s['premises_fol'][j]}")
    if len(s['premises_nl']) > 3:
        print(f"  ... +{len(s['premises_nl'])-3} more premises")

Target unique records: train=320, dev=39, test=41

TARGET SAMPLES (format chuan):

--- Record 0 (11 premises) ---
  [0] NL:  Students who have completed the core curriculum and passed the science assessment are qualified for advanced courses.
      FOL: ∀x ((completed_core_curriculum(x) ∧ passed_science_assessment(x)) → qualified_for_advanced_courses(x))
  [1] NL:  Students who are qualified for advanced courses and have completed research methodology are eligible for the international program.
      FOL: ∀x ((qualified_for_advanced_courses(x) ∧ completed_research_methodology(x)) → eligible_for_international_program(x))
  [2] NL:  Students who have passed the language proficiency exam are eligible for the international program.
      FOL: ∀x (passed_language_proficiency(x) → eligible_for_international_program(x))
  ... +8 more premises

--- Record 1 (11 premises) ---
  [0] NL:  Students who have completed the core curriculum and passed the science assessment are qualified for advanced 

## 5. Thong ke so sanh

In [26]:
# So sanh 2 bo data
avg_premises_target = sum(len(r["premises_nl"]) for r in target_train) / len(target_train)
avg_premises_malls = 1.0  # MALLS luon 1 premise

xor_label = "Has XOR (⊕)"

print(f"{'Metric':<35} {'Target':>10} {'MALLS v0.1':>12}")
print("-" * 59)
print(f"{'Total samples (gop)':<35} {len(target_train)+len(target_dev)+len(target_test):>10} {len(malls_converted):>12}")
print(f"{'Avg premises/record':<35} {avg_premises_target:>10.1f} {avg_premises_malls:>12.1f}")
print(f"{'Naming: CamelCase':<35} {'mixed':>10} {'99.9%':>12}")
print(f"{'Naming: snake_case':<35} {'mixed':>10} {'0.1%':>12}")
print(f"{'Has constants':<35} {'Yes':>10} {'Yes':>12}")
print(f"{xor_label:<35} {'Rare':>10} {has_xor:>12}")

Metric                                  Target   MALLS v0.1
-----------------------------------------------------------
Total samples (gop)                        400        28284
Avg premises/record                       10.4          1.0
Naming: CamelCase                        mixed        99.9%
Naming: snake_case                       mixed         0.1%
Has constants                              Yes          Yes
Has XOR (⊕)                               Rare         1777


## 6. Luu MALLS v0.1 da chuan hoa (gop train+test)

Gop train+test thanh 1 file duy nhat. Ko can do hieu suat Stage 1 — chi can "khoi dong" cho model hieu FOL, metric cuoi do tren Target (Stage 2).

In [27]:
# Luu MALLS v0.1 da chuan hoa — gop train+test thanh 1 file
malls_output_path = OUTPUT_DIR / "malls_v01_normalized.json"

# Chi luu premises_nl va premises_fol (bo source)
malls_save = [
    {"premises_nl": r["premises_nl"], "premises_fol": r["premises_fol"]}
    for r in malls_converted
]

with open(malls_output_path, "w", encoding="utf-8") as f:
    json.dump(malls_save, f, ensure_ascii=False, indent=2)

size_mb = malls_output_path.stat().st_size / 1024 / 1024
print(f"Saved: {malls_output_path}")
print(f"Records: {len(malls_save)} (train {len(malls_train_converted)} + test {len(malls_test_converted)})")
print(f"File size: {size_mb:.1f} MB")

Saved: E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\data\processed\malls_v01_normalized.json
Records: 28284 (train 27284 + test 1000)
File size: 7.7 MB


## 7. Chien luoc Train: 2-Stage Curriculum Learning

### Tai sao KHONG nen gom 1 lan?
- MALLS v0.1: 28K samples vs Target: ~320 records (train)
- Ratio ~88:1 → Target data bi 'chim', model hoc convention MALLS (CamelCase, single-premise)
- Target co multi-premise, mixed naming → model can hoc rieng

### 2-Stage plan:

```
Stage 1: Pretrain on MALLS v0.1 (general NL->FOL)
  - Data: malls_v01_normalized.json (28,284 samples, gop train+test)
  - Ko can eval set — muc tieu chi la "khoi dong" cho model hieu FOL
  - Epochs: 2-3
  - LR: 2e-4
  - Output: checkpoint_stage1/

Stage 2: Fine-tune on Target (domain-specific)
  - Data: data/processed/train.csv (target dataset)
  - Base: checkpoint_stage1/ (NOT base Qwen)
  - Epochs: 15-25 (voi early stopping patience=5)
  - LR: 1e-4 (thap hon Stage 1 de khong 'quen' MALLS)
  - Output: final_lora/
```

### Loi ich:
1. Stage 1: Model hoc cau truc FOL chung (quantifiers, connectives, predicates)
2. Stage 2: Model tinh chinh cho format cu the (JSON, multi-premise, naming style)
3. Giam overfit vi model da co 'nen tang' FOL tu 28K samples

In [28]:
# Quick sanity check: verify normalized MALLS v0.1 can be loaded back
with open(malls_output_path, "r", encoding="utf-8") as f:
    check = json.load(f)

print(f"Reload check: {len(check)} records")
print(f"Sample 0: {check[0]}")
print(f"Sample -1 (last): {check[-1]}")
print()
print("Ready for 2-stage training!")

Reload check: 28284 records
Sample 0: {'premises_nl': ['A film can be a drama, have a long runtime, and win multiple awards, or it can be a comedy, have a shorter runtime, and be a box office success.'], 'premises_fol': ['∃x (Film(x) ∧ ((Drama(x) ∧ LongRuntime(x) ∧ MultipleAwards(x)) ∨ (Comedy(x) ∧ ShorterRuntime(x) ∧ BoxOfficeSuccess(x))))']}
Sample -1 (last): {'premises_nl': ['Astronomers use telescopes to observe celestial objects and gather data, helping to expand our understanding of the universe.'], 'premises_fol': ['∀x∀y (Astronomer(x) ∧ Telescope(y) → (ObservesCelestialObjects(x, y) ∧ GathersData(x, y) ∧ ExpandsUnderstandingOfUniverse(x)))']}

Ready for 2-stage training!
